# KG1 v73 — GRPO TRL Stage-2 (Colab Pro+ A100)

## Framework: priyanlc/autoresearch-sft-grpo + dtdo90 rewards

**Bombas**:
- 6 rewards compostos (priyanlc): correctness + format + reasoning + category_bonus + single_box + final_line
- KL beta=0.01, num_generations=4, LR=5e-6
- Foco: bit_manip 3-input + cryptarithm + equation_guess (gaps huikang)
- Memory: ~36-38GB A100 (vllm rollouts otimizados)

## Pré-requisito: V73 SFT adapter (FASE 2 completa) em Drive ou HF

## Score esperado: 0.86 → 0.87 (P=70%+)

In [ ]:
# Cell 1: Setup + clone repo
import torch, subprocess, os, sys
r = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
print(r.stdout[:800])

from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))

%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q --no-deps 'trl>=0.16' peft>=0.18.1 accelerate bitsandbytes
%pip install -q vllm 'transformers>=4.55' datasets liger-kernel

# Clone repo (para src/competition_utils.py) usando os.system para parseability
if not os.path.exists('/content/kg1'):
    rc = os.system('git clone https://github.com/FELIPEACASTRO/KG1.git /content/kg1 2>/dev/null')
    if rc != 0:
        print('WARN: repo clone falhou, usando inline fallback')
sys.path.insert(0, '/content/kg1/src')
sys.path.insert(0, '/content/kg1')

from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_KEY')
except Exception:
    HF_TOKEN = userdata.get('HF_TOKEN', '')
assert HF_TOKEN.startswith('hf_'), 'Configure HF_KEY no Colab Secrets'
os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
# Cell 2: Load V73 SFT adapter (FASE 2) como ponto de partida
from unsloth import FastLanguageModel

MAX_SEQ = 4096
model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)

# Carregar V73 SFT adapter como base
from peft import PeftModel
V73_SFT_PATH = '/content/drive/MyDrive/kg1_v73_unsloth_moe/final_adapter'
if not os.path.exists(V73_SFT_PATH):
    # Fallback: download from HF
    from huggingface_hub import snapshot_download
    V73_SFT_PATH = snapshot_download(
        'felipesp1983/kg1-nemotron-lora-v73-unsloth-moe',
        token=HF_TOKEN, allow_patterns=['final/*']
    ) + '/final'

model = PeftModel.from_pretrained(model, V73_SFT_PATH, adapter_name='sft', is_trainable=True)
model.set_adapter('sft')
print(f'Loaded V73 SFT from {V73_SFT_PATH}')

In [ ]:
# Cell 3: Reward functions (TRL 0.16+ kwargs vem das colunas do dataset)
import re

# Inline competition_utils (fallback se import do repo falhou)
try:
    from competition_utils import extract_final_answer, verify
    print('Using competition_utils from repo')
except ImportError:
    print('Using inline extract_final_answer/verify (fallback)')
    BOXED_INNER = re.compile(r'\\boxed\{([^{}]+)\}')
    def extract_final_answer(text):
        if not isinstance(text, str): return None
        m = BOXED_INNER.findall(text)
        return m[-1].strip() if m else None
    def verify(answer, predicted):
        if not predicted: return False
        a, p = str(answer).strip(), str(predicted).strip()
        if re.fullmatch(r'[01]+', a):
            return a == p
        return a == p

BOXED_RE = re.compile(r'\\boxed\{([^{}]+)\}')
FINAL_LINE_RE = re.compile(r'(?:Answer|Final|Result|Resposta)[:\s]+([^\n]+)', re.I)

W_CORRECTNESS = 1.0
W_FORMAT = 0.3
W_REASONING = 0.15
W_CATEGORY_BONUS = 0.2
W_SINGLE_BOX = 0.08
W_FINAL_LINE = 0.02

# CRITICO: TRL passa kwargs com NOME EXATO da coluna do dataset
# Dataset (pos to_grpo) tem colunas: prompt, answer (singular), family (singular)

def _to_text(c):
    return c if isinstance(c, str) else (c[0]['content'] if isinstance(c, list) else str(c))

def _normalize_family(fam):
    '''Normaliza family para matching consistente (case + spaces).'''
    return str(fam).lower().replace(' ', '_').replace('-', '_')

def reward_correctness(prompts, completions, answer, **kwargs):
    rewards = []
    for c, a in zip(completions, answer):
        pred = extract_final_answer(_to_text(c))
        rewards.append(W_CORRECTNESS if verify(str(a), str(pred)) else 0.0)
    return rewards

def reward_format(prompts, completions, **kwargs):
    return [W_FORMAT if BOXED_RE.search(_to_text(c)) else 0.0 for c in completions]

def reward_single_box(prompts, completions, **kwargs):
    return [W_SINGLE_BOX if len(BOXED_RE.findall(_to_text(c))) == 1 else 0.0 for c in completions]

def reward_final_line(prompts, completions, **kwargs):
    return [W_FINAL_LINE if FINAL_LINE_RE.search(_to_text(c)) else 0.0 for c in completions]

def reward_reasoning(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        L = len(_to_text(c))
        if 500 <= L <= 3000:
            rewards.append(W_REASONING)
        elif L > 3000:
            rewards.append(W_REASONING * 0.5)
        else:
            rewards.append(0.0)
    return rewards

# Markers por keyword de familia (robusto a naming variants)
FAMILY_MARKERS = {
    'bit':         ['XOR', 'AND', 'OR', 'NOT', 'binary', '01'],
    'cipher':      ['substitution', 'mapping', 'decrypt', 'letter'],
    'encryption':  ['substitution', 'mapping', 'decrypt', 'letter'],
    'crypt':       ['letter', 'digit', 'maps'],
    'equation':    ['operator', 'arithmetic', '+', '-', '*', '/'],
    'gravity':     ['g =', 'd =', 't =', 'gravitational'],
    'unit':        ['multiply', 'convert', 'km', 'mph'],
    'numeral':     ['roman', 'decimal'],
}

def reward_category_bonus(prompts, completions, family, **kwargs):
    rewards = []
    for c, fam in zip(completions, family):
        fam_norm = _normalize_family(fam)
        # Pega primeiro keyword que da match
        ms = []
        for kw, markers in FAMILY_MARKERS.items():
            if kw in fam_norm:
                ms = markers
                break
        if not ms:
            rewards.append(0.0)
            continue
        text = _to_text(c).lower()
        hits = sum(1 for m in ms if m.lower() in text)
        rewards.append(W_CATEGORY_BONUS * min(1.0, hits / len(ms)))
    return rewards

REWARD_FUNCS = [reward_correctness, reward_format, reward_single_box,
                reward_final_line, reward_reasoning, reward_category_bonus]
print(f'Loaded {len(REWARD_FUNCS)} reward functions')


In [ ]:
# Cell 4: Dataset GRPO - foco em hard examples (filtro ROBUSTO)
from datasets import load_dataset

ds_full = load_dataset('felipesp1983/kg1-nemotron-training',
                       data_files='data/sft_v70_huikang_full.jsonl',
                       split='train', token=HF_TOKEN)
print(f'Full dataset: {len(ds_full)} | columns: {ds_full.column_names}')

# Diagnostico: quais categorias existem no dataset?
_cats = {}
for ex in ds_full.select(range(min(500, len(ds_full)))):
    c = ex.get('category', 'unknown')
    _cats[c] = _cats.get(c, 0) + 1
print('Categorias (amostra 500):')
for c, n in sorted(_cats.items(), key=lambda x: -x[1]):
    print(f'  {c!r}: {n}')

# HARD families: normalizacao robusta
# Kaggle oficial usa: Bit Manipulation, Equation Transformation, Text Encryption, etc.
# huikang pode ter usado: bit_manipulation, equation_numeric_deduce, cryptarithm_*, etc.
HARD_KEYWORDS = ['bit', 'crypt', 'equation', 'cipher', 'encryption']

def _is_hard(cat):
    if not cat:
        return False
    cat_l = str(cat).lower().replace(' ', '_')
    return any(kw in cat_l for kw in HARD_KEYWORDS)

ds_hard = ds_full.filter(lambda x: _is_hard(x.get('category', '')), num_proc=4)
print(f'Hard examples (bit + crypt + equation + cipher): {len(ds_hard)} de {len(ds_full)}')

# Safety: se filtro trouxer 0, usa dataset full
if len(ds_hard) == 0:
    print('WARN: filtro hard = 0, usando full dataset')
    ds_hard = ds_full

N_GRPO = min(600, len(ds_hard))
ds_grpo_raw = ds_hard.shuffle(seed=42).select(range(N_GRPO))

def to_grpo(ex):
    msgs = ex['messages']
    user_msg = next((m['content'] for m in msgs if m['role'] == 'user'), '')
    return {
        'prompt': user_msg,
        'answer': str(ex.get('answer', '')),
        'family': str(ex.get('category', 'unknown')),
    }

ds_grpo = ds_grpo_raw.map(to_grpo, num_proc=4, remove_columns=ds_grpo_raw.column_names)
print(f'GRPO ready: {len(ds_grpo)} prompts | columns: {ds_grpo.column_names}')
print(f'Sample prompt[:100]: {ds_grpo[0]["prompt"][:100]!r}')
print(f'Sample answer: {ds_grpo[0]["answer"][:50]!r}')
print(f'Sample family: {ds_grpo[0]["family"]}')


In [ ]:
# Cell 5: GRPO Training (TRL 0.16+ API)
from trl import GRPOTrainer, GRPOConfig

CKPT_DIR = '/content/drive/MyDrive/kg1_v73_grpo'
os.makedirs(CKPT_DIR, exist_ok=True)

grpo_args = GRPOConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_prompt_length=2048,
    max_completion_length=4096,
    learning_rate=5e-6,
    beta=0.01,
    num_train_epochs=1,
    save_steps=50,
    logging_steps=5,
    bf16=True,
    report_to='none',
    push_to_hub=False,
    seed=42,
    use_vllm=False,
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=REWARD_FUNCS,
    args=grpo_args,
    train_dataset=ds_grpo,
    processing_class=tok,
)

resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        resume = True
        print(f'Resuming from existing checkpoint')

trainer.train(resume_from_checkpoint=resume)
trainer.save_model(f'{CKPT_DIR}/final_grpo')
print('GRPO done')


In [ ]:
# Cell 6: Upload V73-GRPO adapter
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v73-grpo'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=f'{CKPT_DIR}/final_grpo', repo_id=REPO_ID, path_in_repo='final')
print(f'Uploaded V73-GRPO to {REPO_ID}')